# Messinian Salinity Crisis: desiccation vs deep-water scenario

Updated for the refactored `primus_paleo` (November 2026). What changed with respect to the previous version:

* **Radiogenic background**: `integrate_fission_spectrum` + `integrate_background_neutron_spectrum` are replaced by one
  `integrate_radiogenic`, returning `{'fission_fragments', 'fission_n', 'alpha_n', 'total'}`. It is computed for each
  scenario (the two samples have different ages).
* **Track groups**: every integral is run with `nucleus='groups'`, so each result also carries the split into
  heavy / medium / light recoils, ready for `plot_track_groups`. The previous total is the `'total_no_light_ions'` entry.
* **Neutrinos** (new here): under ~1 km of halite and brine the cosmic-ray muons are strongly attenuated and the
  atmospheric neutrinos are not, so they are added explicitly (`integrate_atmospheric_neutrinos`).
* **Zenith distribution**: cosmic rays reach the sample along slanted paths weighted by the MCEq zenith distribution
  of each species (bug fix: before, all directions had equal weight in cos(zenith)). Expect fewer tracks near the
  surface and more at tens to hundreds of m.w.e. than with the old code; the neutrino rate also uses the zenith
  distribution.
* **Etching**: `slice_spectrum_etch` returns the expected pit counts, with the resolution set by the objective
  (`magnification`).

The first run builds the MCEq response matrix of the atmosphere once (`Response_*_zen_*.npz` in the flux data folder);
later runs read it.

In [ ]:
import numpy as np
from matplotlib import pyplot as plt

import primus_paleo as pp
from primus_paleo import (FluxHistory, Paleodetector, plot_track_groups, resolution_limit_nm, slice_spectrum_etch)

SUM = pp.TOTAL_NO_LIGHT_IONS   # the total used before ('total_no_light_ions': light ions excluded)

## Scenarios

Unchanged. The overburden dictionaries are now validated: unknown keys or phase lists of different lengths raise.

In [ ]:
# Local time t [kyr] = 5970 - age [ka]: t = 0 at 5.97 Ma (base of the Main Halite), 640 at 5.33 Ma, 5970 today.
# Brine/water depth above the depositional surface (the free parameters of the scenario):
total_age_kyr_A = 5970
flood_time_A = 640.

D0      = 1000.   # at 5.97 Ma, when the sample forms                      [m]
D1      = 800.    # at 5.60 Ma, end of stage 1, before the drawdown        [m]
D_acme  = 300.    # at 5.55 Ma, stage-2 drawdown acme; kept until 5.33 Ma  [m]
D_flood = 2000.   # just after the Zanclean reflooding, 5.33 Ma            [m]
D_today = 1644.   # today (Leviathan-1 water depth)                        [m]

deep_water_base = {
    "initial_depth": D0,
    "initial_density_g_cm3": 1.2,
    "start_time_continuous_kyr": [0.,    0.,              370.,                 370.,  640.,   640.],
    "end_time_continuous_kyr":   [370.,  370.,            420.,                 640.,  5970.,  5970.],
    "rate_continuous":           [2.6,   (D1 - D0) / 370, (D_acme - D1) / 50,   2.7,   0.085,  (D_today - D_flood) / 5330],
    "density_continuous_g_cm3":  [2.16,  1.2,             1.2,                  2.3,   2.1,    1.03],
    "time_discrete_kyr":      [flood_time_A,     flood_time_A],
    "overburden_discrete":    [-D_acme,  D_flood],
    "density_discrete_g_cm3": [1.2,      1.03],
}

In [ ]:
total_age_kyr_B = 5600
flood_time_B = 270

z_floor = 2000.    # depth of the brine surface below global sea level during the desiccation [m]
air_mwe = 10.33 * (np.exp(z_floor / 8400.) - 1.0)   # extra atmosphere above it, ~2.8 m.w.e. for 2 km
D_brine = 5.       # brine depth during the halite deposition, 5.60-5.55 Ma [m]
D_lake  = 20.      # lake depth during stage 3, Upper Gypsum / Lago-Mare, 5.55-5.33 Ma [m]
D_flood = 2000.    # seawater depth just after the Zanclean reflooding, 5.33 Ma [m]
D_today = 1644.    # today (Leviathan-1 water depth) [m]
H_halite, H_IE, H_AE, H_PQ = 960., 480., 230., 450.   # Main Halite, Interbedded Ev., Argillaceous Ev., Plio-Quaternary [m]

desiccation_base = {
    "initial_depth": air_mwe + 1.2 * D_brine, "initial_density_g_cm3": 1.0,   # extra air + brine, in m.w.e.
    "start_time_continuous_kyr": [0.,              50.,           180.,         270.,           270.],
    "end_time_continuous_kyr":   [50.,             180.,          270.,         5600.,          5600.],
    "rate_continuous":           [H_halite / 50.,  H_IE / 130.,   H_AE / 90.,   H_PQ / 5330.,   (D_today - D_flood) / 5330.],
    "density_continuous_g_cm3":  [2.16,            2.3,           2.3,          2.1,            1.03],
    "time_discrete_kyr":      [50.,              flood_time_B,    flood_time_B,      flood_time_B],
    "overburden_discrete":    [D_lake - D_brine, -D_lake,  -air_mwe,  D_flood],
    "density_discrete_g_cm3": [1.2,              1.2,      1.0,       1.03],
}

In [ ]:
mineral_name = "Halite"

sample_area_cm2 = 1 * 2
magnification = 10          # objective used for the etched samples: sets the smallest pit seen

In [ ]:
flux_history_simple = FluxHistory()
flux_history_SN100 = FluxHistory(events=[{"start_time_kyr": -total_age_kyr_B - 25., "template": {"kind": "SN", "distance_pc": 100.}}])

In [ ]:
mineral = Paleodetector(mineral_name, total_age_kyr=total_age_kyr_B, overburden_history=desiccation_base, flux_history=flux_history_simple)
mineral_deepwater = Paleodetector(mineral_name, total_age_kyr=total_age_kyr_A, overburden_history=deep_water_base, flux_history=flux_history_simple)

mineral.describe_track_groups()     # which recoils count as heavy / medium / light in halite

## Overburden histories

`_overburden_interpolator` gives the overburden [m.w.e.] at any local time: a quick check that the two scenarios are
what we mean before integrating.

In [ ]:
fig, ax = plt.subplots(figsize=[7, 4])
for det, label, flood in ((mineral, 'Desiccation', flood_time_B), (mineral_deepwater, 'Deep water', flood_time_A)):
    t = np.linspace(0., det.total_age_kyr, 4000)
    ax.plot(t, det._overburden_interpolator(t), label=label)
    ax.axvline(flood, color=ax.lines[-1].get_color(), linestyle=':', linewidth=1)
ax.set_xlabel('Local time since the sample formed (kyr)')
ax.set_ylabel('Overburden (m.w.e.)')
ax.set_yscale('log')
ax.set_xscale('symlog', linthresh=100)
ax.legend(fontsize='small')
plt.show()

## Track spectra

Radiogenic background over the whole age of each sample; cosmic rays and neutrinos in the exposure window up to the
reflooding (and after it, for the desiccation sample). `flux_history=` is passed explicitly every time, since it stays
set on the detector after a call.

In [ ]:
x_bins, radiogenic_B = mineral.integrate_radiogenic(nucleus='groups')
x_bins, radiogenic_A = mineral_deepwater.integrate_radiogenic(nucleus='groups')

In [ ]:
window_B, window_A, window_post = [0., flood_time_B], [0., flood_time_A], [flood_time_B, total_age_kyr_B]

x_bins, tracks_simple = mineral.integrate_all_particles(exposure_window_kyr=window_B, nucleus='groups',
                                                        flux_history=flux_history_simple)
x_bins, tracks_simple_deepwater = mineral_deepwater.integrate_all_particles(exposure_window_kyr=window_A, nucleus='groups',
                                                                            flux_history=flux_history_simple)
x_bins, tracks_post = mineral.integrate_all_particles(exposure_window_kyr=window_post, nucleus='groups',
                                                      flux_history=flux_history_simple)

In [ ]:
x_bins, neutrinos_B = mineral.integrate_atmospheric_neutrinos(exposure_window_kyr=window_B, nucleus='groups',
                                                              flux_history=flux_history_simple)
x_bins, neutrinos_A = mineral_deepwater.integrate_atmospheric_neutrinos(exposure_window_kyr=window_A, nucleus='groups',
                                                                        flux_history=flux_history_simple)

In [ ]:
x_bins, tracks_SN100 = mineral.integrate_all_particles(exposure_window_kyr=window_B, nucleus='groups',
                                                       flux_history=flux_history_SN100)

In [ ]:
def total(result):
    """Total of an integrate_* result with nucleus='groups' (the same quantity as the old ['total'])."""
    return result['total'][SUM]

fig, ax = plt.subplots(figsize=[7, 5])
x_um = x_bins / 1000

ax.stairs(total(tracks_simple), x_um, label='Simple scenario, desiccation')
ax.stairs(total(tracks_SN100), x_um, color='C2', label='SN100 scenario, desiccation')
ax.stairs(total(tracks_simple_deepwater) * 1e4, x_um, color='C1', label='Simple scenario, deep water x 1e4')
ax.stairs(total(tracks_post), x_um, color='C3', label='Post-event (after the reflooding)')
ax.stairs(total(neutrinos_B), x_um, color='C4', linestyle='dotted', label='Atmospheric neutrinos, desiccation')

ax.stairs(radiogenic_B['fission_fragments'][SUM], x_um, color='C5', linestyle='dashed', label='Fission tracks')
ax.stairs(radiogenic_B['fission_n'][SUM] + radiogenic_B['alpha_n'][SUM], x_um, color='C6', linestyle='dashed',
          label='Radiogenic-neutron tracks')

ax.set_xlabel(r"Track Length ($\mu$m)")
ax.set_ylabel(r"Number of Tracks (mg$^{-1}$)")
ax.set_yscale("log")
ax.set_ylim(1e-4, 1e3)
ax.set_xlim(0, 25)
ax.legend(fontsize='small')
plt.show()

In [ ]:
n = {label: np.sum(total(result)) for label, result in (
    ('cosmic B', tracks_simple), ('cosmic A', tracks_simple_deepwater), ('SN100 B', tracks_SN100),
    ('nu B', neutrinos_B), ('nu A', neutrinos_A), ('post B', tracks_post))}
print(f"Tracks per mg, desiccation (0-{flood_time_B} kyr): cosmic rays {n['cosmic B']:.3g}, neutrinos {n['nu B']:.3g}; "
      f"after the reflooding {n['post B']:.3g}")
print(f"Tracks per mg, deep water  (0-{flood_time_A:g} kyr): cosmic rays {n['cosmic A']:.3g}, neutrinos {n['nu A']:.3g}")
if n['cosmic A'] > 0:
    print(f"Desiccation / deep water, cosmic rays: {n['cosmic B'] / n['cosmic A']:.3g}")
else:
    print("Deep water: no cosmic-ray tracks (overburden beyond the StdRock depth tables): neutrinos only.")
print(f"Desiccation / deep water, cosmic rays + neutrinos: {(n['cosmic B'] + n['nu B']) / (n['cosmic A'] + n['nu A']):.3g}")
print(f"SN100 excess in the desiccation sample: {n['SN100 B'] / n['cosmic B'] - 1:+.2%}")

## Who makes the tracks

`plot_track_groups` takes any dictionary of components. First by source (each source summed over its track groups),
one figure per scenario, then by track group within the main sources.

In [ ]:
def by_source(cosmic, neutrinos, radiogenic):
    """Components of one sample, each the 'total_no_light_ions' sum: cosmic-ray species, neutrinos, radiogenic."""
    out = {species: v[SUM] for species, v in cosmic.items() if species != 'total'}
    out['neutrinos'] = neutrinos['total'][SUM]
    out['fission'] = radiogenic['fission_fragments'][SUM]
    out['radiogenic n'] = radiogenic['fission_n'][SUM] + radiogenic['alpha_n'][SUM]
    out['total'] = sum(out.values())
    return out

sources_B = by_source(tracks_simple, neutrinos_B, radiogenic_B)
sources_A = by_source(tracks_simple_deepwater, neutrinos_A, radiogenic_A)
keys = [k for k in sources_B if k != 'total']

for label, sources in (('Desiccation (0-270 kyr)', sources_B), ('Deep water (0-640 kyr)', sources_A)):
    fig, ax = plot_track_groups(sources, x_bins, style='lines', keys=keys, total='total', logy=True, y_decades=7)
    ax.set_title(label)
    plt.show()

In [ ]:
fig, axes = plot_track_groups({'Cosmic rays, desiccation': tracks_simple['total'],
                               'Cosmic rays, SN100': tracks_SN100['total'],
                               'Neutrinos, desiccation': neutrinos_B['total'],
                               'Radiogenic': radiogenic_B['total']},
                              x_bins, style='panels', logy=True, y_decades=6)
plt.show()

## Recoil energies (optional)

The same integrals binned in recoil energy instead of track length, per recoiling nuclide (`nucleus='all'`): useful to
see which nuclides carry the signal, independently of the range tables.

In [ ]:
e_bins, recoils_B = mineral.integrate_all_particles(exposure_window_kyr=window_B, nucleus='all',
                                                   observable='recoil_energy', flux_history=flux_history_simple)
nuclides = recoils_B['total']
heaviest = sorted((k for k in nuclides if k not in pp.TRACK_SUMS), key=lambda k: -np.sum(nuclides[k]))[:6]
fig, ax = plot_track_groups(nuclides, e_bins, style='lines', keys=heaviest, total='total', observable='recoil_energy',
                            logy=True, y_decades=6)
plt.show()

## Etched spectra

Expected pit counts on the analysed area after HF etching, as seen at `magnification`; the dotted line is the
smallest pit width resolved there (`resolution_limit_nm`). `realisation=True` would give one Poisson draw instead.

In [ ]:
density = mineral.config["density_g_cm3"]
etch = dict(sample_area_cm2=sample_area_cm2, sample_density_g_cm3=density, etch_type='HF', magnification=magnification,
            x_bins=x_bins)

sliced_bkg = slice_spectrum_etch(radiogenic_B['total'][SUM], **etch)
sliced_bkg_deepwater = slice_spectrum_etch(radiogenic_A['total'][SUM] + total(neutrinos_A), **etch)
sliced_simple = slice_spectrum_etch(total(tracks_simple) + total(neutrinos_B), **etch)
sliced_SN100 = slice_spectrum_etch(total(tracks_SN100) + total(neutrinos_B), **etch)
sliced_simple_deepwater = slice_spectrum_etch(total(tracks_simple_deepwater), **etch)

In [ ]:
fig, ax = plt.subplots(figsize=[7, 5])

ax.stairs(sliced_bkg, x_um, color='C5', label='Radiogenic only (desiccation sample)')
ax.stairs(sliced_simple_deepwater + sliced_bkg_deepwater, x_um, color='C1', label='Deep-water scenario')
ax.stairs(sliced_simple + sliced_bkg, x_um, color='C0', label='Simple scenario, desiccation')
ax.stairs(sliced_SN100 + sliced_bkg, x_um, color='C2', label='SN100 scenario, desiccation')
ax.axvline(resolution_limit_nm(magnification) / 1000, color='grey', linestyle=':', label=f'Resolution at {magnification}x')

ax.set_xlabel(r"Pit length ($\mu$m)")
ax.set_ylabel(r"Expected pits per bin")
ax.set_yscale("log")
ax.set_ylim(1e0, 1e2)
ax.set_xlim(0, 5)
ax.legend(fontsize='small')
plt.show()

# The expected pit count is linear in the track spectrum, so the SN excess is etched on its own: its Monte Carlo
# error is then a fraction of the excess, not of the much larger total.
excess_SN100 = slice_spectrum_etch(np.clip(total(tracks_SN100) - total(tracks_simple), 0., None), **etch)
n_bkg, n_simple, n_excess = np.sum(sliced_bkg), np.sum(sliced_simple + sliced_bkg), np.sum(excess_SN100)
print(f"Pits on {sample_area_cm2} cm2: radiogenic {n_bkg:.0f}, desiccation {n_simple:.0f}, SN100 excess {n_excess:.1f}")
print(f"SN100 excess: {n_excess / np.sqrt(n_simple):.2f} sigma (Poisson, counts only)")